# 01 — EDA + Weight of Evidence / Information Value
**Owner:** Panashe  |  **Phase:** 1  |  **Date:** May 11

Objectives:
- Understand data distributions and default patterns
- Compute WoE/IV for every feature to drive feature selection
- Critical data quality checks (duplicates, date logic, outliers, train/test drift)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys, pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve()))

from src.data_loader import load_train, load_test
from src.woe_iv import compute_woe_iv, iv_summary, plot_woe

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
%matplotlib inline

ModuleNotFoundError: No module named 'src'

In [ ]:
train = load_train()
test  = load_test()
print(f"Train: {train.shape}  |  Test: {test.shape}")
train.head()

## 1. Target Distribution

In [ ]:
counts = train["Target"].value_counts()
print(counts)
print(f"\nDefault rate: {counts[1]/len(train):.1%}")

## 2. Missing Value Analysis

In [ ]:
missing = train.isna().mean().sort_values(ascending=False) * 100
print(missing[missing > 0])

## 3. Default Rate by Categorical Feature

In [ ]:
cat_cols = ["product_code","payment_frequency","loan_purpose","client_gender",
             "marital_status","employment_sector","collateral_type",
             "disbursement_channel","province"]
for col in cat_cols:
    rates = train.groupby(col)["Target"].mean().sort_values(ascending=False)
    print(f"\n--- {col} ---\n{rates.to_string()}")

## 4. Numeric Feature Distributions by Target

In [ ]:
num_cols = ["amount_usd","annual_rate_pct","term_months","monthly_income_usd",
             "existing_obligations","num_dependents","months_at_employer"]
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, col in zip(axes.flat, num_cols):
    for t, color in [(0, "#4575b4"), (1, "#d73027")]:
        train[train["Target"]==t][col].dropna().hist(ax=ax, alpha=0.6, color=color, bins=40, label=str(t))
    ax.set_title(col); ax.legend(title="Default")
plt.tight_layout()

## 5. Correlation Matrix

In [ ]:
corr = train[num_cols + ["Target"]].corr()
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0, ax=ax)
plt.title("Correlation Matrix")
plt.tight_layout()

## 6. Data Quality Checks

In [ ]:
# Duplicates
print(f"Duplicate rows: {train.duplicated().sum()}")

# ID format: XX99999
import re
bad_ids = train["ID"][~train["ID"].str.match(r'^[A-Z]{2}\d{5}$', na=False)]
print(f"Invalid IDs: {len(bad_ids)}")

# Date logic
print("\nDate logic violations:")
print(f"  disbursed < approved:          {(train.date_disbursed < train.date_approved).sum()}")
print(f"  first_payment < disbursed:     {(train.first_payment_due < train.date_disbursed).sum()}")
print(f"  maturity < first_payment:      {(train.maturity_date < train.first_payment_due).sum()}")
under18 = (train.client_dob > (train.date_approved - pd.DateOffset(years=18)))
print(f"  borrower under 18 at approval: {under18.sum()}")

In [ ]:
# Outliers (IQR)
for col in ["amount_usd","annual_rate_pct","monthly_income_usd"]:
    q1, q3 = train[col].quantile(0.25), train[col].quantile(0.75)
    iqr = q3 - q1
    n = ((train[col] < q1-1.5*iqr) | (train[col] > q3+1.5*iqr)).sum()
    print(f"{col}: {n} outliers ({n/len(train):.1%})")

# Train/Test drift
print("\nKS test (p<0.05 = drift):")
for col in ["amount_usd","annual_rate_pct","term_months","monthly_income_usd"]:
    stat, p = stats.ks_2samp(train[col].dropna(), test[col].dropna())
    print(f"  {col}: p={p:.4f}  {'DRIFT' if p<0.05 else 'ok'}")

## 7. WoE / IV Analysis

In [ ]:
all_features = num_cols + cat_cols
cat_features = cat_cols
iv_df = iv_summary(train, all_features, "Target", cat_features=cat_features)
print(iv_df.to_string())
to_drop = iv_df[iv_df["iv"] < 0.02]["feature"].tolist()
print(f"\nDrop candidates (IV < 0.02): {to_drop}")

In [ ]:
for feat in iv_df.head(3)["feature"]:
    is_cat = feat in cat_cols
    woe_df, iv = compute_woe_iv(train, feat, "Target", cat=is_cat)
    fig, ax = plt.subplots(figsize=(10, 3))
    plot_woe(woe_df, f"{feat}  (IV={iv:.3f})", ax=ax)
    plt.savefig(f"reports/figures/woe_{feat}.png", dpi=120, bbox_inches="tight")
    plt.show()